# Лабораторная работа 2. Основы Pandas
Михайлов Яков Витальевич, группа 1521741.

Задачи 1–6 из «Тема 2. Pandas — основы». Исходные данные сотрудников находятся в data.py.
Выполняйте ячейки последовательно. После очистки данных пересчитываем total_comp, чтобы учитывать заполненные зарплаты.

In [1]:
from pathlib import Path
import runpy
import numpy as np
import pandas as pd

LAB_DIR = Path.cwd()
if not (LAB_DIR / "data.py").exists():
    LAB_DIR = LAB_DIR / "labs/02_numpy_pandas"
data = runpy.run_path(str(LAB_DIR / "data.py"))["data"]
print("Pandas:", pd.__version__)

Pandas: 2.2.3


## 1. Series, DataFrame и первичный осмотр
### 1.1–1.3

In [2]:
series = pd.Series([10, 20, 30, 40], index=["a", "b", "c", "d"], name="numbers")
df = pd.DataFrame(data)
print("Series:\n", series)
print("Первые строки:\n", df.head())
print("Последние строки:\n", df.tail())
print("Информация:")
df.info()
print("Описательная статистика:\n", df.describe())
assert df.shape == (8, 5)

Series:
 a    10
b    20
c    30
d    40
Name: numbers, dtype: int64
Первые строки:
       name department   salary   bonus  hire_date
0    Alice         IT  80000.0  5000.0 2020-01-15
1      Bob         HR  60000.0  3000.0 2019-03-22
2  Charlie         IT  90000.0     NaN 2021-07-01
3    Diana    Finance  75000.0  4000.0 2018-11-05
4      Eve         HR      NaN  2000.0 2022-02-10
Последние строки:
     name department   salary   bonus  hire_date
3  Diana    Finance  75000.0  4000.0 2018-11-05
4    Eve         HR      NaN  2000.0 2022-02-10
5  Frank         IT  85000.0  6000.0 2020-09-30
6  Grace    Finance  70000.0  3500.0 2019-12-12
7  Henry         HR  62000.0     NaN 2021-05-20
Информация:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   name        8 non-null      object        
 1   department  8 non-null      object        
 2  

## 2. Индексация и фильтрация
### 2.1–2.3

In [3]:
print("IT: имена и зарплаты:\n", df.loc[df["department"] == "IT", ["name", "salary"]])
print("Первые три строки и два столбца:\n", df.iloc[:3, :2])
print("Зарплата выше 70000:\n", df.loc[df["salary"] > 70000])

IT: имена и зарплаты:
       name   salary
0    Alice  80000.0
2  Charlie  90000.0
5    Frank  85000.0
Первые три строки и два столбца:
       name department
0    Alice         IT
1      Bob         HR
2  Charlie         IT
Зарплата выше 70000:
       name department   salary   bonus  hire_date
0    Alice         IT  80000.0  5000.0 2020-01-15
2  Charlie         IT  90000.0     NaN 2021-07-01
3    Diana    Finance  75000.0  4000.0 2018-11-05
5    Frank         IT  85000.0  6000.0 2020-09-30


### 2.4–2.5. Общая компенсация и удаление hire_date

In [4]:
df["total_comp"] = df["salary"] + df["bonus"].fillna(0)
df = df.drop(columns="hire_date")
print(df.to_string(index=False))
assert "hire_date" not in df.columns
assert df.loc[df["name"] == "Charlie", "total_comp"].iloc[0] == 90000

   name department  salary  bonus  total_comp
  Alice         IT 80000.0 5000.0     85000.0
    Bob         HR 60000.0 3000.0     63000.0
Charlie         IT 90000.0    NaN     90000.0
  Diana    Finance 75000.0 4000.0     79000.0
    Eve         HR     NaN 2000.0         NaN
  Frank         IT 85000.0 6000.0     91000.0
  Grace    Finance 70000.0 3500.0     73500.0
  Henry         HR 62000.0    NaN     62000.0


## 3. Пропущенные значения
### 3.1–3.4

In [5]:
print("Пропуски до обработки:\n", df.isna().sum())
department_mean = df.groupby("department")["salary"].transform("mean")
df["salary"] = df["salary"].fillna(department_mean)
df["bonus"] = df["bonus"].fillna(0)
rows_before = len(df)
df = df.dropna(subset=["salary"]).copy()
# total_comp зависит от salary, поэтому обновляем его после заполнения.
df["total_comp"] = df["salary"] + df["bonus"]
print("Удалено строк:", rows_before - len(df))
print("Пропуски после обработки:\n", df.isna().sum())
print(df.to_string(index=False))
assert df.loc[df["name"] == "Eve", "salary"].iloc[0] == 61000
assert df.loc[df["name"] == "Eve", "total_comp"].iloc[0] == 63000
assert not df.isna().any().any()

Пропуски до обработки:
 name          0
department    0
salary        1
bonus         2
total_comp    1
dtype: int64
Удалено строк: 0
Пропуски после обработки:
 name          0
department    0
salary        0
bonus         0
total_comp    0
dtype: int64
   name department  salary  bonus  total_comp
  Alice         IT 80000.0 5000.0     85000.0
    Bob         HR 60000.0 3000.0     63000.0
Charlie         IT 90000.0    0.0     90000.0
  Diana    Finance 75000.0 4000.0     79000.0
    Eve         HR 61000.0 2000.0     63000.0
  Frank         IT 85000.0 6000.0     91000.0
  Grace    Finance 70000.0 3500.0     73500.0
  Henry         HR 62000.0    0.0     62000.0


Зарплата Eve заполнена средним по HR: (60000 + 62000) / 2 = 61000. Если бы в отделе отсутствовали все зарплаты, среднее тоже было бы NaN и соответствующие строки удалились бы.

## 4. Группировка и агрегация
### 4.1–4.4

In [6]:
mean_salary = df.groupby("department")["salary"].mean()
summary = df.groupby("department").agg(mean_salary=("salary", "mean"), max_bonus=("bonus", "max"), employees=("name", "size"))
pivot = df.pivot_table(index="department", values="salary", aggfunc="mean")
ordered = mean_salary.sort_values(ascending=False)
print("Средняя зарплата:\n", mean_salary)
print("Агрегация:\n", summary)
print("Сводная таблица:\n", pivot)
print("Отделы по убыванию средней зарплаты:\n", ordered)
assert mean_salary.to_dict() == {"Finance": 72500.0, "HR": 61000.0, "IT": 85000.0}

Средняя зарплата:
 department
Finance    72500.0
HR         61000.0
IT         85000.0
Name: salary, dtype: float64
Агрегация:
             mean_salary  max_bonus  employees
department                                   
Finance         72500.0     4000.0          2
HR              61000.0     3000.0          3
IT              85000.0     6000.0          3
Сводная таблица:
              salary
department         
Finance     72500.0
HR          61000.0
IT          85000.0
Отделы по убыванию средней зарплаты:
 department
IT         85000.0
Finance    72500.0
HR         61000.0
Name: salary, dtype: float64


## 5. Объединение данных
### 5.1–5.2
Имена руководителей и новый сотрудник — учебные примеры: в исходном файле их нет.

In [7]:
managers = pd.DataFrame({"department": ["IT", "HR", "Finance"], "manager": ["Ivan", "Olga", "Sergey"]})
df = df.merge(managers, on="department", how="left", validate="many_to_one")
print(managers.to_string(index=False))
print("После left join:\n", df.to_string(index=False))
assert len(df) == 8 and df["manager"].notna().all()

department manager
        IT    Ivan
        HR    Olga
   Finance  Sergey
После left join:
    name department  salary  bonus  total_comp manager
  Alice         IT 80000.0 5000.0     85000.0    Ivan
    Bob         HR 60000.0 3000.0     63000.0    Olga
Charlie         IT 90000.0    0.0     90000.0    Ivan
  Diana    Finance 75000.0 4000.0     79000.0  Sergey
    Eve         HR 61000.0 2000.0     63000.0    Olga
  Frank         IT 85000.0 6000.0     91000.0    Ivan
  Grace    Finance 70000.0 3500.0     73500.0  Sergey
  Henry         HR 62000.0    0.0     62000.0    Olga


### 5.3–5.4. Новый сотрудник, concat и сброс индекса

In [8]:
new_employee = pd.DataFrame({"name": ["Irene"], "department": ["IT"], "salary": [95000.0], "bonus": [7000.0]})
new_employee["total_comp"] = new_employee["salary"] + new_employee["bonus"]
new_employee = new_employee.merge(managers, on="department", how="left", validate="many_to_one")
df = pd.concat([df, new_employee], axis=0)
print("Индексы после concat:", df.index.tolist())
df = df.reset_index(drop=True)
print("После сброса индекса:\n", df.to_string(index=False))
assert len(df) == 9 and df.index.tolist() == list(range(9))
np.testing.assert_allclose(df["total_comp"], df["salary"] + df["bonus"])

Индексы после concat: [0, 1, 2, 3, 4, 5, 6, 7, 0]
После сброса индекса:
    name department  salary  bonus  total_comp manager
  Alice         IT 80000.0 5000.0     85000.0    Ivan
    Bob         HR 60000.0 3000.0     63000.0    Olga
Charlie         IT 90000.0    0.0     90000.0    Ivan
  Diana    Finance 75000.0 4000.0     79000.0  Sergey
    Eve         HR 61000.0 2000.0     63000.0    Olga
  Frank         IT 85000.0 6000.0     91000.0    Ivan
  Grace    Finance 70000.0 3500.0     73500.0  Sergey
  Henry         HR 62000.0    0.0     62000.0    Olga
  Irene         IT 95000.0 7000.0    102000.0    Ivan


## 6. Комплексный мини-анализ
### 6.1–6.3
Используется итоговая таблица из 9 сотрудников, включая добавленную Irene.

In [9]:
top3 = df.nlargest(3, "total_comp")[["name", "department", "total_comp"]]
means = df.groupby("department").agg(mean_salary=("salary", "mean"), mean_bonus=("bonus", "mean"))
shares = df.assign(above_75000=df["salary"] > 75000).groupby("department")["above_75000"].mean().rename("share_above_75000")
print("Топ-3 по общей компенсации:\n", top3.to_string(index=False))
print("Средняя зарплата и бонус:\n", means)
print("Доля зарплат строго выше 75000:\n", shares.to_frame().assign(percent=shares * 100))
assert top3["name"].tolist() == ["Irene", "Frank", "Charlie"]
assert shares.to_dict() == {"Finance": 0.0, "HR": 0.0, "IT": 1.0}
assert means.loc["IT", "mean_salary"] == 87500
assert means.loc["IT", "mean_bonus"] == 4500

Топ-3 по общей компенсации:
    name department  total_comp
  Irene         IT    102000.0
  Frank         IT     91000.0
Charlie         IT     90000.0
Средняя зарплата и бонус:
             mean_salary   mean_bonus
department                          
Finance         72500.0  3750.000000
HR              61000.0  1666.666667
IT              87500.0  4500.000000
Доля зарплат строго выше 75000:
             share_above_75000  percent
department                            
Finance                   0.0      0.0
HR                        0.0      0.0
IT                        1.0    100.0


**Вывод:** данные очищены, таблицы объединены, общая компенсация рассчитана. После добавления Irene средняя зарплата IT — 87500, средний бонус — 4500. Первые три места по общей компенсации: Irene (102000), Frank (91000), Charlie (90000). Доля сотрудников с зарплатой строго выше 75000: IT — 100%, HR и Finance — 0%.